# Construção do MVP - Regressão Logística

## O que Encontrar neste Notebook?

# Importando as Bibliotecas

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)

from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)


from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Carregamentos dos Dados

In [2]:
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


# KPI's de Performance

## KPI Técnico - Recall (Sensibilidade)

O principal KPI técnico escolhido, dado a natureza do nosso problema de negócio, foi o **Recall.** Aqui o objetivo é simples, queremos identificar o seguinte cenário: **de todos os clientes que iam churnar, quantos o modelo conseguiu identificar corretamente.** 

Queremos reduzir ao máximo os casos de Falsos Negativos que representam clientes que churnaram e não agimos, resultando na perda de receita.

Essa métrica é crítica para o negócio e permite responder perguntas como: ***"Quanto churn eu estou deixando passar?***"

Para a experimentação vamos considerar modelos com recall > 0.70.

## KPI's Técnicos Secundários

Outras métricas técnicas serão utilizadas para critério de possíveis desempates entre modelos e também para uma melhor interpretação do desempenho nosso modelo:

**`auc`** - avalia o quão bem o modelo consegue separar os clientes que vão churnar dos que não vão (buscamos algo acima de 0,70);

**`precision`** - avalia quantos clientes que foram marcado como churn de fato churnaram. Essa métrica tem um impacto direta no caso de falsos positivos e estabelece um trade-off com `recall`. Ela representa o custo desnecessário com retenção de clientes que não vão churnar;

**`f1-score`** - é uma métrica de equilibrio entre `precision` e `recall`. Ela consegue balancear custo de retenção e perda de clientes, porém não considera o valor financeiro do cliente (assume custos fp e fn, aproximadamente iguais, o que quase nunca é verdade). Como já possuímos métricas financeiras, essa métrica será para uso secundário. Ela nos auxiliará na identificação do quão eficiente o modelo está conseguindo capturar churn.


## KPI de Negócio (definir)

As métricas técnicas presumem que todos os clientes são iguais do ponto de vista financeiro. E, sabemos que isso não ocorre dentro do nosso cenário real do negócio. Dessa forma, o uso dos KPIs econômicos nos permite priorizar clientes de maior valor, reduzir perda financeira real, balancear a relação custo-benefício e tomar decisões mais assertivas com base no negócio.

Em outras palavras, o modelo não só prevê churn, ele prioriza clientes com maior impacto financeiro e maximiza o retorno da retenção. Abaixo temos alguns KPIs econômicos que podemos utilizar para avaliar nosso modelo do ponto de vista do negócio:

- Receita Protegida (Somatória de Montlhy Charges dos TP);
- Receita Perdida (CLTV dos FN);
- Custo de Retenção (FP * Custo_unitário);
- Valor Liquído ou ROI - retorno real gerado pelo modelo (Receita Protegida - Custo de Retenção);
- CLTV Médio dos TP (qualidade dos clientes capturados);
- Taxa de Captura de Valor: quanto do valor em risco conseguimos capturar? (Receita Protegida / (Receita Protegida + Receita Perdida));
- ROI da Retenção ((receita protegida - custo)/custo)

# Pré-Processamento dos Dados

## Sanity Check

In [3]:
# Informações gerais sobre o dataset
print("=== INFORMAÇÕES GERAIS DO DATASET ===\n")
print(df.info())

# Análise de valores ausentes
print("=== ANÁLISE DE MISSING VALUES ===\n")

missing_values = pd.DataFrame({
    'Coluna': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

if len(missing_values) > 0:
    print(missing_values)

# Valores Duplicados
print("\n=== ANÁLISE DE VALORES DUPLICADOS ===\n")
duplicate_count = df.duplicated().sum()
print(f"Total de linhas duplicadas: {duplicate_count}")



=== INFORMAÇÕES GERAIS DO DATASET ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null 

## Dropando Variáveis

In [4]:
drop_cols = [
    'CustomerID',
    'Country',
    'State',
    'Lat Long',
    'Churn Label',
    'Churn Reason',
    'CLTV',
    'Latitude',
    'Longitude',
    'City',
    'Churn Score',
    'Zip Code',
    'Count'
]

df.drop(columns=drop_cols, inplace=True)

In [5]:
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,1


## Tratamento dos Dados

In [6]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [7]:
df['Total Charges'].isnull().sum()

np.int64(11)

In [8]:
df[df['Total Charges'].isna()][['Tenure Months', 'Monthly Charges']]

,Tenure Months,Monthly Charges
2234,0,52.55
2438,0,20.25
2568,0,80.85
2667,0,25.75
2856,0,56.05
4331,0,19.85
4687,0,25.35
5104,0,20.00
5719,0,19.70
6772,0,73.35


In [9]:
df['Total Charges'] = df['Total Charges'].fillna(0)

## Codificando as Variáveis

In [10]:
# Separando as Variáveis
target = 'Churn Value'
num_vars = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
cat_vars = [col for col in df.columns if df[col].dtype in ['str', 'object', 'category']]

print(f"Variáveis numéricas: {num_vars}")
print(f"Variáveis categóricas: {cat_vars}")

Variáveis numéricas: ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Value']
Variáveis categóricas: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


In [11]:
# dummificando as variáveis categóricas
dummy_vars = pd.get_dummies(df[cat_vars], drop_first=True, dtype=int)
dummy_vars.head()

,Gender_Male,Senior Citizen_Yes,Partner_Yes,Dependents_Yes,Phone Service_Yes,Multiple Lines_No phone service,Multiple Lines_Yes,Internet Service_Fiber optic,Internet Service_No,Online Security_No internet service,...,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
1,0,0,0,1,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
2,0,0,0,1,1,0,1,1,0,0,...,0,1,0,1,0,0,1,0,1,0
3,0,0,1,1,1,0,1,1,0,0,...,0,1,0,1,0,0,1,0,1,0
4,1,0,0,1,1,0,1,1,0,0,...,0,1,0,1,0,0,1,0,0,0


In [12]:
# concatenando as variáveis numéricas e dummificadas
df_final = pd.concat([df[num_vars], dummy_vars], axis=1)
df_final.head()

,Tenure Months,Monthly Charges,Total Charges,Churn Value,Gender_Male,Senior Citizen_Yes,Partner_Yes,Dependents_Yes,Phone Service_Yes,Multiple Lines_No phone service,...,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,2,53.85,108.15,1,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
1,2,70.70,151.65,1,0,0,0,1,1,0,...,0,0,0,0,0,0,1,0,1,0
2,8,99.65,820.50,1,0,0,0,1,1,0,...,0,1,0,1,0,0,1,0,1,0
3,28,104.80,3046.05,1,0,0,1,1,1,0,...,0,1,0,1,0,0,1,0,1,0
4,49,103.70,5036.30,1,1,0,0,1,1,0,...,0,1,0,1,0,0,1,0,0,0


## Estratégia de Validação (StratifiedKFold)

In [13]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

## Separando as Bases

In [14]:
X = df_final.drop(columns=[target])
y = df_final[target]


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y  
)

# Baseline (DummyClassifier)

In [15]:
# Modelo baseline (Dummy)
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)


# Predições
y_train_pred = dummy.predict(X_train)
y_test_pred = dummy.predict(X_test)

# Para AUC (probabilidade)
y_train_proba = dummy.predict_proba(X_train)[:, 1]
y_test_proba = dummy.predict_proba(X_test)[:, 1]


# Métricas - Treino
metrics_train = {
    'recall': recall_score(y_train, y_train_pred),
    'precision': precision_score(y_train, y_train_pred, zero_division=0),
    'f1_score': f1_score(y_train, y_train_pred, zero_division=0),
    'auc': roc_auc_score(y_train, y_train_proba)
}


# Métricas - Teste
metrics_test = {
    'recall': recall_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred, zero_division=0),
    'f1_score': f1_score(y_test, y_test_pred, zero_division=0),
    'auc': roc_auc_score(y_test, y_test_proba)
}


# Overfitting (Recall)
recall_train = metrics_train['recall']
recall_test = metrics_test['recall']

# Evita divisão por zero
if recall_train == 0:
    overfitting_recall = 0.0
else:
    overfitting_recall = ((recall_train - recall_test) / recall_train) * 100


# Log estruturado
log_metrics = {
    'train': metrics_train,
    'test': metrics_test,
    'overfitting_recall_%': overfitting_recall
}


# Print formatado
print("===== BASELINE - DUMMY =====\n")

print(">> TREINO")
for k, v in metrics_train.items():
    print(f"{k}: {v:.4f}")

print("\n>> TESTE")
for k, v in metrics_test.items():
    print(f"{k}: {v:.4f}")

print("\n>> OVERFITTING (Recall)")
print(f"{overfitting_recall:.2f}%")


===== BASELINE - DUMMY =====

>> TREINO
recall: 0.0000
precision: 0.0000
f1_score: 0.0000
auc: 0.5000

>> TESTE
recall: 0.0000
precision: 0.0000
f1_score: 0.0000
auc: 0.5000

>> OVERFITTING (Recall)
0.00%


# Pipeline da Regressão Logística

In [16]:
# Pipeline
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])



scoring = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)


# Agregação CV
cv_metrics = {
    'train': {
        'recall': np.mean(cv_results['train_recall']),
        'precision': np.mean(cv_results['train_precision']),
        'f1_score': np.mean(cv_results['train_f1']),
        'auc': np.mean(cv_results['train_roc_auc'])
    },
    'validation': {
        'recall': np.mean(cv_results['test_recall']),
        'precision': np.mean(cv_results['test_precision']),
        'f1_score': np.mean(cv_results['test_f1']),
        'auc': np.mean(cv_results['test_roc_auc'])
    }
}


# Fit final no treino
pipeline.fit(X_train, y_train)


# Predições
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

y_train_proba = pipeline.predict_proba(X_train)[:, 1]
y_test_proba = pipeline.predict_proba(X_test)[:, 1]


# Métricas - Treino
metrics_train = {
    'recall': recall_score(y_train, y_train_pred),
    'precision': precision_score(y_train, y_train_pred),
    'f1_score': f1_score(y_train, y_train_pred),
    'auc': roc_auc_score(y_train, y_train_proba)
}


# Métricas - Teste
metrics_test = {
    'recall': recall_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred),
    'f1_score': f1_score(y_test, y_test_pred),
    'auc': roc_auc_score(y_test, y_test_proba)
}


# Overfitting (Recall)
recall_train = metrics_train['recall']
recall_test = metrics_test['recall']

if recall_train == 0:
    overfitting_recall = 0.0
else:
    overfitting_recall = ((recall_train - recall_test) / recall_train) * 100


# Log estruturado
log_metrics = {
    'cv': cv_metrics,
    'train': metrics_train,
    'test': metrics_test,
    'overfitting_recall_%': overfitting_recall
}


# Print formatado
print("===== LOGISTIC REGRESSION (PIPELINE + CV) =====\n")

print(">> CROSS VALIDATION (MÉDIA)")
for split in ['train', 'validation']:
    print(f"\n[{split.upper()}]")
    for k, v in cv_metrics[split].items():
        print(f"{k}: {v:.4f}")

print("\n>> TREINO (FULL FIT)")
for k, v in metrics_train.items():
    print(f"{k}: {v:.4f}")

print("\n>> TESTE")
for k, v in metrics_test.items():
    print(f"{k}: {v:.4f}")

print("\n>> OVERFITTING (Recall)")
print(f"{overfitting_recall:.2f}%")


===== LOGISTIC REGRESSION (PIPELINE + CV) =====

>> CROSS VALIDATION (MÉDIA)

[TRAIN]
recall: 0.8185
precision: 0.5328
f1_score: 0.6454
auc: 0.8624

[VALIDATION]
recall: 0.8134
precision: 0.5308
f1_score: 0.6422
auc: 0.8573

>> TREINO (FULL FIT)
recall: 0.8211
precision: 0.5325
f1_score: 0.6460
auc: 0.8623

>> TESTE
recall: 0.7950
precision: 0.5156
f1_score: 0.6255
auc: 0.8531

>> OVERFITTING (Recall)
3.18%


# Pipeline MLP - PyTorch

## Configurações da Rede Neural

In [17]:
# Configuração
EPOCHS = 30
BATCH_SIZE = 32
LR = 0.001  # taxa de aprendizado
WD = 1e-5   # weight decay (regularização)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
THRESHOLD = 0.5   


## Arquitetura do MLP

In [18]:
# Modelo MLP
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),            # Camada de entrada para oculta
            nn.ReLU()                                    # Função de ativação ReLU
        )

        self.out = nn.Linear(hidden_dim, output_dim)     # Camada de saída

    def forward(self, X):

        features = self.features(X)
        output = self.out(features)

        return output

## Funções Auxiliares

In [19]:
def compute_metrics(y_true, y_proba, threshold=0.5):
    y_true = y_true.ravel()
    y_pred = (y_proba >= threshold).astype(int)

    return {
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_proba)
    }

# Média CV
def avg_metrics(metrics_list):
    return {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0]}

## Fluxo de Treinamento e Validação

In [24]:

# Função de treino
def train_model(model, dataloader, optimizer, criterion, device=DEVICE):
    model.train()
    
    total_loss = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE).unsqueeze(1)

        optimizer.zero_grad()
        
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


# Validação
def evaluate(model, dataloader, criterion, device=DEVICE):
    model.eval()
    total_loss = 0
    preds = []
    targets = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE).unsqueeze(1)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            probs = torch.sigmoid(outputs)

            preds.append(probs.cpu())
            targets.append(y_batch.cpu())

            total_loss += loss.item()

    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()

    metrics = compute_metrics(targets, preds, threshold=THRESHOLD)

    return total_loss / len(dataloader), metrics

## Validação Cruzada

In [36]:
# Cross-validation
cv_train_metrics = []
cv_val_metrics = []

X_train = np.asarray(X_train)
y_train = np.asarray(y_train)

for train_idx, val_idx in cv.split(X_train, y_train):
    
    # Split dos dados
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    # Escalonamento dos Dados
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)

    # calculo dos pesos para a classe positiva (churn) e negativa (não churn)
    pos = y_tr.sum()
    neg = len(y_tr) - pos

    pos_weight = torch.tensor(
        [neg / pos if pos > 0 else 1.0],
        dtype=torch.float32
    ).to(DEVICE)
    
    # tensor
    train_dataset = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32)
    )

    val_dataset   = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32)
    )

    # dataloader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    
    # Modelo
    model = MLP(input_dim=X_train.shape[1]).to(DEVICE)
    
    # Otimizador e critério
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


    #Treino
    for epoch in range(10):
        train_loss = train_model(model, train_loader, optimizer, criterion)
        val_loss, _ = evaluate(model, val_loader, criterion)

        print(f"Epoch {epoch+1} | Train_loss: {train_loss:.4f} | Val_loss: {val_loss:.4f}")

    # métricas finais do fold
    _, train_metrics = evaluate(model, train_loader, criterion)
    _, val_metrics = evaluate(model, val_loader, criterion)

    cv_train_metrics.append(train_metrics)
    cv_val_metrics.append(val_metrics)



# Agregação CV
cv_train_mean = avg_metrics(cv_train_metrics)
cv_val_mean = avg_metrics(cv_val_metrics)


cv_results_df = pd.DataFrame({
    "Train": cv_train_mean,
    "Validation": cv_val_mean
})

cv_results_df["Gap (%)"] = (
    (cv_results_df["Train"] - cv_results_df["Validation"]) 
    / cv_results_df["Train"]
) * 100

print("\n===== CV RESULTS =====")
display(cv_results_df.round(4))

Epoch 1 | Train_loss: 0.7985 | Val_loss: 0.7209
Epoch 2 | Train_loss: 0.7028 | Val_loss: 0.7020
Epoch 3 | Train_loss: 0.6824 | Val_loss: 0.6977
Epoch 4 | Train_loss: 0.6744 | Val_loss: 0.6955
Epoch 5 | Train_loss: 0.6655 | Val_loss: 0.6980
Epoch 6 | Train_loss: 0.6590 | Val_loss: 0.6971
Epoch 7 | Train_loss: 0.6558 | Val_loss: 0.6942
Epoch 8 | Train_loss: 0.6522 | Val_loss: 0.6964
Epoch 9 | Train_loss: 0.6489 | Val_loss: 0.6958
Epoch 10 | Train_loss: 0.6430 | Val_loss: 0.6963
Epoch 1 | Train_loss: 0.7988 | Val_loss: 0.7701
Epoch 2 | Train_loss: 0.6973 | Val_loss: 0.7506
Epoch 3 | Train_loss: 0.6775 | Val_loss: 0.7477
Epoch 4 | Train_loss: 0.6677 | Val_loss: 0.7368
Epoch 5 | Train_loss: 0.6607 | Val_loss: 0.7390
Epoch 6 | Train_loss: 0.6544 | Val_loss: 0.7525
Epoch 7 | Train_loss: 0.6491 | Val_loss: 0.7313
Epoch 8 | Train_loss: 0.6454 | Val_loss: 0.7360
Epoch 9 | Train_loss: 0.6421 | Val_loss: 0.7333
Epoch 10 | Train_loss: 0.6390 | Val_loss: 0.7404
Epoch 1 | Train_loss: 0.7857 | Val_los

,Train,Validation,Gap (%)
recall,0.8269,0.7996,3.2937
precision,0.5509,0.5344,2.9905
f1,0.6610,0.6399,3.1914
auc,0.8773,0.8583,2.1671


## Conclusão

# Persistindo o Modelo

# Persistindo a Base

In [ ]:
# Persistir o dataframe pré-processado
#df_final.to_csv("../data/processed/telco_customer_churn_preprocessed.csv", index=False)

# Próximos Passos

- Prosseguir para Etapa de Experimentação do Modelo de Regressão, realizando feature engineering;